# MRL-0809 Stage-4 Retry Launcher

This notebook is a transport-only launcher for the single Founder-authorized successor-v2 Stage-4 retry. It does not change canonical `main`.

It fail-stops unless `origin/main` is exactly `2d8af295b86144a06ce6fe9aad741fed07c2f5d7` with tree `5cc4c44d83aa9e51c63c7283a3b037dfa2e3a7ff`, the runtime is Google Colab with exactly one Tesla T4 (15360 MiB), `/proc/sys/kernel/random/boot_id` supplies the fresh runtime-instance identity used by the canonical pre-probe evidence, and the repository's accepted retry authority gate passes.

The notebook uses only synthetic input, no training, no scientific corpus, no offload, no model/revision substitution, and no paid fallback. It packages receipts/logs/markers while excluding large model snapshot directories, then downloads a compact evidence ZIP even when the governed driver fails.

In [ ]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import traceback
import zipfile

from google.colab import files

CANONICAL_SHA = "2d8af295b86144a06ce6fe9aad741fed07c2f5d7"
CANONICAL_TREE = "5cc4c44d83aa9e51c63c7283a3b037dfa2e3a7ff"
REPO_URL = "https://github.com/TheHalfMoon/MESC.git"
REPO = Path("/content/MESC")
CUSTODY = Path("/content/mrl0809-v2-retry-1-custody")
LOG = Path("/content/mrl0809-stage4-retry-launch.log")
SUMMARY = Path("/content/mrl0809-stage4-retry-launch-summary.json")
MANIFEST = Path("/content/mrl0809-stage4-retry-evidence-manifest.json")
BUNDLE = Path("/content/mrl0809-stage4-retry-evidence.zip")
MIN_FREE_BYTES = 25_025_488_695

summary: dict[str, object] = {}
failure: BaseException | None = None


def write_log(message: str) -> None:
    with LOG.open("a", encoding="utf-8") as handle:
        handle.write(message.rstrip() + "\n")


def run(argv: list[str], *, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    write_log("$ " + " ".join(argv))
    with LOG.open("a", encoding="utf-8") as handle:
        completed = subprocess.run(
            argv,
            cwd=None if cwd is None else str(cwd),
            check=False,
            text=True,
            stdout=handle,
            stderr=subprocess.STDOUT,
        )
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed with exit {completed.returncode}: {argv[0]}")
    return completed


def text(argv: list[str], *, cwd: Path | None = None) -> str:
    return subprocess.check_output(
        argv,
        cwd=None if cwd is None else str(cwd),
        text=True,
        encoding="utf-8",
    ).strip()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


try:
    import google.colab  # noqa: F401

    release = os.environ.get("COLAB_RELEASE_TAG", "").strip()
    if not release:
        raise RuntimeError("COLAB_RELEASE_TAG is missing; not a qualified Colab runtime")

    gpu_lines = text(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total",
            "--format=csv,noheader,nounits",
        ]
    ).splitlines()
    if len(gpu_lines) != 1:
        raise RuntimeError("exactly one CUDA GPU is required")
    gpu_name, gpu_memory = [part.strip() for part in gpu_lines[0].rsplit(",", 1)]
    if gpu_name != "Tesla T4" or int(gpu_memory) != 15360:
        raise RuntimeError(f"STANDARD Tesla T4 required; observed {gpu_name} / {gpu_memory} MiB")

    boot_id_path = Path("/proc/sys/kernel/random/boot_id")
    if not boot_id_path.is_file():
        raise RuntimeError("runtime boot identity file is unavailable")
    runtime_id = boot_id_path.read_text(encoding="ascii").strip()
    parts = runtime_id.split("-")
    if (
        len(parts) != 5
        or [len(part) for part in parts] != [8, 4, 4, 4, 12]
        or any(not part or any(ch not in "0123456789abcdef" for ch in part.lower()) for part in parts)
    ):
        raise RuntimeError("runtime boot identity is malformed")

    os.environ["MESC_COLAB_RUNTIME_ID"] = "manual-colab-boot:" + runtime_id
    os.environ["MESC_MRL0809_ZERO_COST_ATTESTATION"] = "1"
    os.environ.pop("HF_TOKEN", None)
    os.environ.pop("HUGGING_FACE_HUB_TOKEN", None)

    summary.update(
        {
            "colab_release_tag": release,
            "gpu_memory_mib": int(gpu_memory),
            "gpu_name": gpu_name,
            "provider_execution_id_sha256": hashlib.sha256(runtime_id.encode("ascii")).hexdigest(),
            "provider_execution_id_source": "/proc/sys/kernel/random/boot_id",
            "zero_cost_attestation": True,
        }
    )

    if REPO.exists():
        shutil.rmtree(REPO)
    if CUSTODY.exists():
        raise RuntimeError("custody directory already exists; refusing possible retry")

    run(["git", "clone", REPO_URL, str(REPO)])
    remote_main = text(["git", "ls-remote", "origin", "refs/heads/main"], cwd=REPO).split()[0]
    if remote_main != CANONICAL_SHA:
        raise RuntimeError(f"origin/main moved before launch: {remote_main}")
    run(["git", "checkout", "--detach", CANONICAL_SHA], cwd=REPO)
    if text(["git", "rev-parse", "HEAD"], cwd=REPO) != CANONICAL_SHA:
        raise RuntimeError("canonical SHA checkout mismatch")
    if text(["git", "rev-parse", "HEAD^{tree}"], cwd=REPO) != CANONICAL_TREE:
        raise RuntimeError("canonical tree mismatch")
    if text(["git", "status", "--porcelain"], cwd=REPO):
        raise RuntimeError("canonical checkout is not clean")

    run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "uv==0.12.22"])
    run(["uv", "python", "install", "3.11"], cwd=REPO)
    run(["uv", "sync", "--no-dev", "--extra", "rq1-runtime-feasibility", "--frozen"], cwd=REPO)
    run(["uv", "cache", "clean"], cwd=REPO)
    run(["sudo", "apt-get", "update"])
    run(["sudo", "apt-get", "install", "-y", "bubblewrap"])

    free_bytes = shutil.disk_usage("/content").free
    summary["free_bytes_before_stage"] = free_bytes
    if free_bytes < MIN_FREE_BYTES:
        raise RuntimeError(f"insufficient free storage before staging: {free_bytes}")

    driver = [
        str(REPO / ".venv/bin/python"),
        str(REPO / "scripts/mesc_mrl_0809_stage4_v2_retry_driver.py"),
        "--repository-root",
        str(REPO),
        "--custody",
        str(CUSTODY),
        "--python-executable",
        str(REPO / ".venv/bin/python"),
        "--expected-canonical-revision",
        CANONICAL_SHA,
    ]
    completed = run(driver, cwd=REPO, check=False)
    summary["driver_returncode"] = completed.returncode
    if completed.returncode != 0:
        raise RuntimeError(f"authorized Stage-4 driver failed with exit {completed.returncode}")

    summary["launcher_result"] = "PASS"
except BaseException as exc:
    failure = exc
    summary["launcher_result"] = "FAIL"
    summary["failure_type"] = type(exc).__name__
    summary["failure_message"] = str(exc)
    write_log(traceback.format_exc())
finally:
    summary["canonical_sha"] = CANONICAL_SHA
    summary["canonical_tree"] = CANONICAL_TREE
    summary["generated_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()
    markers = []
    if CUSTODY.exists():
        markers = sorted(str(path.relative_to(CUSTODY)) for path in CUSTODY.rglob("*.probe-start.json"))
    summary["probe_start_markers"] = markers
    SUMMARY.write_text(json.dumps(summary, sort_keys=True, indent=2) + "\n", encoding="utf-8")

    included: list[tuple[Path, str]] = [(LOG, "launcher/launch.log"), (SUMMARY, "launcher/summary.json")]
    if CUSTODY.exists():
        for path in sorted(CUSTODY.rglob("*")):
            if not path.is_file():
                continue
            relative = path.relative_to(CUSTODY)
            if any(part.endswith("-snapshot") for part in relative.parts):
                continue
            included.append((path, "custody/" + relative.as_posix()))

    manifest_rows = []
    for path, archive_name in included:
        if path.exists():
            manifest_rows.append(
                {
                    "archive_path": archive_name,
                    "sha256": sha256_file(path),
                    "size_bytes": path.stat().st_size,
                }
            )
    MANIFEST.write_text(json.dumps(manifest_rows, sort_keys=True, indent=2) + "\n", encoding="utf-8")
    included.append((MANIFEST, "launcher/evidence-manifest.json"))

    with zipfile.ZipFile(BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path, archive_name in included:
            if path.exists():
                archive.write(path, archive_name)
    print(json.dumps(summary, sort_keys=True, indent=2))
    files.download(str(BUNDLE))

if failure is not None:
    raise RuntimeError("MRL-0809 Stage-4 retry did not PASS; inspect downloaded evidence bundle") from failure